<a href="https://colab.research.google.com/github/saminsiddiqui08-beep/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saminsiddiqui08-beep/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

->
* I selected Lane D (Under-Clicked Visible Pages).
* The ML Task Type is Ranking & Priority Scoring.
* The content team can only review a fixed number of pages each week (e.g. 50). To make sure that their time is used efficiently, the system creates a ranked list of pages based on how much extra traffic each page could realistically gain if improved. The pages with the biggest potential gains are placed at the top of the list, so the team always works on the ones that promise the best results.

In [6]:
import os, sys, subprocess
import pandas as pd
import numpy as np

if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/saminsiddiqui08-beep/flyrank-ml-internship"
    REPO_DIR = "flyrank-ml-internship"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)

# 1. Loading starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 2. Filtering to the visible candidate slice for Lane D (impressions >= 100)
visible_df = df[df["impressions_90d"] >= 100].copy()

print("=== Section 1: Lane D Task Scope ===")
print(f"Total dataset rows: {len(df):,}")
print(f"Visible candidate pool (>=100 impressions): {len(visible_df):,} pages ({len(visible_df)/len(df)*100:.1f}% of total)")


=== Section 1: Lane D Task Scope ===
Total dataset rows: 30,000
Visible candidate pool (>=100 impressions): 22,006 pages (73.4% of total)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

->
* What we predict: We do just look at raw CTR. We calculate the CTR Deficit (how much a page underperforms compared to what’s expected i.e. Expected CTR - Actual CTR) and turn that into Unrealized Opportunity Clicks which is basically the extra clicks a page could gain if it met expectations.


* Observed vs. Proxy: We directly observe clicks, impressions, and average position from logs. The proxy translates those raw outcomes into a business‑actionable measure: “How many extra clicks could this page realistically gain?”

In [7]:
# 1. Computing expected benchmark CTR (grouped by position tier and main intent)
visible_df["expected_ctr_benchmark"] = visible_df.groupby(["position_tier", "main_intent"])["ctr"].transform("median")

# 2. Calculating the CTR Deficit (clipped at 0 so overperforming pages receive 0 deficit)
visible_df["ctr_deficit"] = (visible_df["expected_ctr_benchmark"] - visible_df["ctr"]).clip(lower=0)

# 3. Computing Unrealized Opportunity Clicks
visible_df["unrealized_clicks_proxy"] = (visible_df["ctr_deficit"] / 100.0) * visible_df["impressions_90d"]

print("=== Section 2: Top 5 Highest Opportunity Target Candidates ===")
cols_to_show = ["content_id", "avg_position", "impressions_90d", "ctr", "expected_ctr_benchmark", "ctr_deficit", "unrealized_clicks_proxy"]
print(visible_df.sort_values(by="unrealized_clicks_proxy", ascending=False)[cols_to_show].head(5).to_string(index=False))


=== Section 2: Top 5 Highest Opportunity Target Candidates ===
          content_id  avg_position  impressions_90d  ctr  expected_ctr_benchmark  ctr_deficit  unrealized_clicks_proxy
content_36ff89c8214e           7.3           295097 0.05                    0.22         0.17                 501.6649
content_c8e9d6ab9013           9.7           208678 0.00                    0.22         0.22                 459.0916
content_c84a0ab98e90           7.8           223271 0.03                    0.22         0.19                 424.2149
content_5fe46e04994d           4.2           517715 0.14                    0.22         0.08                 414.1720
content_8451fc6f034d           2.3           272144 0.03                    0.14         0.11                 299.3584


## 3. Success metric

*One metric you can defend. What number means 'good'?*

->
* The metric that I choose is Precision@50 on High-Opportunity Pages.
* Metrics such as $RMSE$ or $R^2$ evaluate predictions across thousands of deep and low-impact pages that content teams will probably never touch. Content teams have a fixed batch capacity of 50 URLs per cycle. Precision@50 directly measures what fraction of those 50 recommendations are genuine high-opportunity targets.
* Normally, fewer than 1% of pages are high‑opportunity (172 out of 22,006). If the model gets at least 35 out of 50 right (Precision@50 ≥ 0.700), that’s a huge improvement over manual inspection.

In [8]:
# High visibility (avg_position <= 20) AND significant uncaptured traffic (>= 50 lost clicks)
visible_df["is_high_opportunity"] = ((visible_df["avg_position"] <= 20) & (visible_df["unrealized_clicks_proxy"] >= 50)).astype(int)

total_high_opp = visible_df["is_high_opportunity"].sum()
base_rate = visible_df["is_high_opportunity"].mean()

# 2. Precision@K evaluation function
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

p50 = precision_at_k(visible_df["unrealized_clicks_proxy"], visible_df["is_high_opportunity"], 50)

print("=== Section 3: Success Metric Baseline ===")
print(f"Total high-opportunity URLs in dataset: {total_high_opp:,}")
print(f"Base opportunity rate: {base_rate*100:.2f}%")
print(f"Target Proxy Precision@50: {p50:.3f} (Captures {int(p50*50)}/50 valid targets)")


=== Section 3: Success Metric Baseline ===
Total high-opportunity URLs in dataset: 172
Base opportunity rate: 0.78%
Target Proxy Precision@50: 1.000 (Captures 50/50 valid targets)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

->

* One row = One unique indexed URL aggregated over a rolling 90-day search performance window.

In [9]:
# Extracting the clean unit of analysis slice
unit_of_analysis = visible_df[[
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "avg_position",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "unrealized_clicks_proxy"
]].copy()

print("=== Section 4: Unit of Analysis DataFrame ===")
print(f"Shape: {unit_of_analysis.shape[0]:,} rows × {unit_of_analysis.shape[1]} columns")
print("\nFirst 5 rows:")
display(unit_of_analysis.head(5))


=== Section 4: Unit of Analysis DataFrame ===
Shape: 22,006 rows × 9 columns

First 5 rows:


,content_id,client_id,content_type,main_intent,avg_position,impressions_90d,clicks_90d,ctr,unrealized_clicks_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,10.6,3803,29,0.76,0.000
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,20.3,15320,7,0.05,1.532
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,36.5,12581,11,0.09,0.000
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,6.2,11751,58,0.49,0.000
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,44.0,19140,24,0.13,0.000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

-> ML beats a fixed rule here because of a couple of reasons:

* The decay of CTR by ranking position is exponential i.e. not linear. Moving from rank 2 to rank 1 yields a much larger expected CTR gain than moving from rank 9 to rank 8. A linear heuristic cannot capture this curve.
* A static threshold (e.g., IF CTR < 0.20%) creates systematic errors - In top_3 positions, informational content has a median CTR of $0.14\%$, while transactional content has a median CTR of $0.29\%$. A fixed rule would incorrectly flag healthy informational pages (false positives) while ignoring underperforming transactional pages (false negatives).
* Expected CTR depends on the non-linear interaction between ranking position, search intent, content format, and commercial competition. Machine learning models learn this multi-dimensional surface directly from data.

In [10]:
# Showing empirical CTR variation across Position Tier and Search Intent
intent_table = visible_df.groupby(["position_tier", "main_intent"])["ctr"].agg(
    Page_Count="count",
    Median_CTR="median",
    Mean_CTR="mean"
).round(3)

print("=== Section 5: CTR Heterogeneity across Position Tier & Intent ===")
display(intent_table.loc[["top_3", "page_1", "striking"]])


=== Section 5: CTR Heterogeneity across Position Tier & Intent ===


Page_Count  Median_CTR  Mean_CTR
position_tier main_intent                                    
top_3         commercial            108       0.190     0.317
              informational         262       0.140     0.250
              transactional         157       0.290     0.407
page_1        commercial           1427       0.220     0.323
              informational        4936       0.220     0.334
              navigational            8       0.325     0.410
              transactional        2006       0.250     0.359
striking      commercial            951       0.150     0.273
              informational        3657       0.150     0.250
              navigational            6       0.460     0.498
              transactional        1153       0.170     0.258

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.